# 3. Symbolic Framwork
Casadi use **compressed column storage** to store matrics

## 3.1 The SX symbolics

In [9]:
from casadi import *

x = SX.sym('x')
x

SX(x)

In [10]:
x2 = SX.sym('x')
x2

SX(x)

In [ ]:
# x and x2 have the same name, but they are different symbolic variables.
print(x+x2)

(x+x)


In [15]:
# vector and matrix
y = SX.sym('y', 5)
z = SX.sym('z', 4, 2)
y, z

(SX([y_0, y_1, y_2, y_3, y_4]),
 SX(
 [[z_0, z_4], 
  [z_1, z_5], 
  [z_2, z_6], 
  [z_3, z_7]]))

In [17]:
# SX instances can form expressions
f = x**2 + 3*x + 2
f = sqrt(f)
f   

SX(sqrt(((sq(x)+(3*x))+2)))

In [22]:
# Constaant SX instances
B1 = SX.zeros(4, 5) # dense matrix
B2 = SX(4,5) # sparse matrix
B3 = SX.eye(4) # sparse identity matrix
B1, B2, B3

(SX(@1=0, 
 [[@1, @1, @1, @1, @1], 
  [@1, @1, @1, @1, @1], 
  [@1, @1, @1, @1, @1], 
  [@1, @1, @1, @1, @1]]),
 SX(
 [[00, 00, 00, 00, 00], 
  [00, 00, 00, 00, 00], 
  [00, 00, 00, 00, 00], 
  [00, 00, 00, 00, 00]]),
 SX(@1=1, 
 [[@1, 00, 00, 00], 
  [00, @1, 00, 00], 
  [00, 00, @1, 00], 
  [00, 00, 00, @1]]))

In [ ]:
A = SX([[1,2],[3,4]])
B = repmat(SX(1), 2, 2) # create a 2x2 matrix filled with 1s, dense by default

C = np.array([[1,2],[3,4]])
C = SX(C)

A, B, C

(SX(
 [[1, 2], 
  [3, 4]]),
 SX(@1=1, 
 [[@1, @1], 
  [@1, @1]]),
 SX(
 [[1, 2], 
  [3, 4]]))

## 3.2 DM
Nonezero elements are numerical values, not symbolic expressions

In [28]:
C = DM(2,3)
C_dense = C.full()
C, C_dense

(DM(
 [[00, 00, 00], 
  [00, 00, 00]]),
 array([[0., 0., 0.],
        [0., 0., 0.]]))

In [29]:
C_array = np.array(C)
print(np.equal(C_array, C_dense))

[[ True  True  True]
 [ True  True  True]]


In [30]:
from scipy.sparse import csc_matrix
C_sparse = C.sparse()
C_sci_sparse = csc_matrix(C)
C_sparse, C_sci_sparse

(<Compressed Sparse Column sparse matrix of dtype 'float64'
 	with 0 stored elements and shape (2, 3)>,
 <Compressed Sparse Column sparse matrix of dtype 'float64'
 	with 0 stored elements and shape (2, 3)>)

## 3.3 The MX symbolics

In [31]:
# SX expresions, restrict to unary and binary operations
x = SX.sym('x', 2, 2)
y = SX.sym('y')
f = 3*x + y
print(f)
print(f.shape)
print(f[0,0])

@1=3, 
[[((@1*x_0)+y), ((@1*x_2)+y)], 
 [((@1*x_1)+y), ((@1*x_3)+y)]]
(2, 2)
((3*x_0)+y)


In [ ]:
# MX, not restricted to unary and binary operations, and input and output can be matrixes
x = MX.sym('x', 2, 2)
y = MX.sym('y')
f = 3*x + y
print(f)
print(f.shape)
print(f[0,0])

((3*x)+y)
(2, 2)
((3*x)+y)[0]


In [33]:
# set MX elements
x = MX.sym('x', 2)
A = MX(2,2)
A[0,0] = x[0]
A[1,1] = x[1] + x[0]
print("A = ", A)

A =  (project((zeros(2x2,1nz)[0] = x[0]))[1] = (x[1]+x[0]))


In [ ]:
# Sparity and matrix multiplication
M = SX([[3,7],[4,5]])
print(M)
M[0,:] = 1
print(M)
M[2]= 2 # from upper left, to lower right, column-wise
print(M)


[[3, 7], 
 [4, 5]]
@1=1, 
[[@1, @1], 
 [4, 5]]

[[1, 2], 
 [4, 5]]


In [ ]:
# Slice copies the data, not a view, so it can be assigned to.
M1 = M[0,:]
M1[:] = 1
print("M = ", M)
print("M1 = ", M1)


[[1, 2], 
 [4, 5]]
@1=1, [[@1, @1]]


In [ ]:
# list access; not as fast as slice
M = SX([[3,7,8,9],[4,5,6,1]])
print(M)
print(M[0,[0,3]])

print(M[[5,-6]]) # from upper left, to lower right, column-wise


[[3, 7, 8, 9], 
 [4, 5, 6, 1]]
[[3, 9]]
[6, 7]


## 3.6 Arithmetic operations

In [45]:
x = SX.sym('x')
y = SX.sym('y', 2,2)
print(x)
print(y)
f = sin(y) - x
print(f)
print(y*y)
print(y@y)
print(y.T)

x

[[y_0, y_2], 
 [y_1, y_3]]

[[(sin(y_0)-x), (sin(y_2)-x)], 
 [(sin(y_1)-x), (sin(y_3)-x)]]

[[sq(y_0), sq(y_2)], 
 [sq(y_1), sq(y_3)]]

[[(sq(y_0)+(y_2*y_1)), ((y_0*y_2)+(y_2*y_3))], 
 [((y_1*y_0)+(y_3*y_1)), ((y_1*y_2)+sq(y_3))]]

[[y_0, y_1], 
 [y_2, y_3]]


In [46]:
# reshape
x = SX.eye(4)
print(reshape(x, 2, 8))

@1=1, 
[[@1, 00, 00, 00, 00, @1, 00, 00], 
 [00, 00, @1, 00, 00, 00, 00, @1]]


In [50]:
# Concatenate
x = SX.sym('x', 5)
y = SX.sym('y', 5)
print(x)
print(y)
v_conc = vertcat(x,y)
print(v_conc)
print(v_conc.shape)
print(horzcat(x,y))

[x_0, x_1, x_2, x_3, x_4]
[y_0, y_1, y_2, y_3, y_4]
[x_0, x_1, x_2, x_3, x_4, y_0, y_1, y_2, y_3, y_4]
(10, 1)

[[x_0, y_0], 
 [x_1, y_1], 
 [x_2, y_2], 
 [x_3, y_3], 
 [x_4, y_4]]


In [ ]:
# Split
x = SX.sym('x', 5,2)
print(x)
w = horzsplit(x, [0,1,2])
print(w)
print("shape of horzsplit: ", w[0].shape, w[1].shape)
print(w[0], w[1])
w = vertsplit(x, [0,3,5]) # why just use 3, and set first and the last index to default value?
print("shape of vertsplit: ", w[0].shape, w[1].shape)
print(w[0], w[1])
# slice could also be used to split, and only in MX, split functions are more efficient.


[[x_0, x_5], 
 [x_1, x_6], 
 [x_2, x_7], 
 [x_3, x_8], 
 [x_4, x_9]]
[SX([x_0, x_1, x_2, x_3, x_4]), SX([x_5, x_6, x_7, x_8, x_9])]
shape of horzsplit:  (5, 1) (5, 1)
[x_0, x_1, x_2, x_3, x_4] [x_5, x_6, x_7, x_8, x_9]
shape of vertsplit:  (3, 2) (2, 2)

[[x_0, x_5], 
 [x_1, x_6], 
 [x_2, x_7]] 
[[x_3, x_8], 
 [x_4, x_9]]
